# Proyek Akhir Praktikum Data Science
## Klasifikasi Tingkat Obesitas Berdasarkan Karakteristik Individu dan Kebiasaan Hidup

**Kelompok 55 — kaum deadline**  
Hasan (123240041) · Nurjelita Sukmawati (123240046) · Haerunisa (123240047) · Puput Septiani (123240263)

**Program Studi Informatika — UPN “Veteran” Yogyakarta**  
**Metode:** klasifikasi multikelas (7 kategori)  
**Sumber:** UCI Machine Learning Repository (2019), salinan CSV pada repository kelompok.

> **PERINGATAN KELAYAKAN:** Menurut metadata resmi UCI, sekitar **77%** observasi dataset ini dihasilkan secara sintetis (WEKA/SMOTE). Panduan Proyek Akhir 2026/2027 halaman 3, ketentuan dataset poin **3(f)** melarang data sintetis. Oleh karena itu, **notebook yang secara teknis lengkap tidak otomatis memenuhi syarat tugas**. Penggunaan dataset ini memerlukan keputusan/izin eksplisit pengampu. Notebook ini tidak menyatakan dataset telah lolos aturan tersebut dan tidak membuat data baru.

**Tujuan:** menyajikan alur Business Understanding → Data Gathering → EDA → Cleaning & Preprocessing → Feature Selection → Modelling → Evaluasi → Konklusi. Ini analisis klasifikasi dataset, **bukan** diagnosis, skrining klinis tervalidasi, atau aplikasi.

**Cara menjalankan:** simpan CSV asli `Obesity_prediction.csv` (2.111 baris) di root repository; simpan notebook di `notebooks/`; instal library dalam `requirements.txt`; jalankan **Run All** dan **simpan semua output**. Notebook akan menolak CSV yang jumlah baris/kolomnya tidak cocok dengan dataset sumber. Jika tidak ada berkas lokal, pembacaan melalui GitHub memerlukan internet.

---
# 1. Business Understanding

### 1.1 Permasalahan
Kategori berat badan dalam dataset mencakup tujuh kelompok, dari berat badan kurang hingga obesitas tipe III. Data mencakup antropometri (tinggi, berat), demografi, kebiasaan makan, dan aktivitas. Pengelompokan ini menarik untuk dilihat sebagai persoalan **klasifikasi multikelas**.

**Masalah analitis:** apakah kategori tersebut dapat dibedakan secara konsisten berdasarkan karakteristik yang tersedia? Seberapa besar hasil model berubah ketika tinggi dan berat badan—yang berhubungan langsung dengan definisi kategori berat badan—tidak disertakan?

### 1.2 Pertanyaan penelitian
1. **P1:** Bagaimana distribusi tujuh kategori obesitas dan karakteristik responden?
2. **P2:** Bagaimana pola *asosiasi deskriptif* antara riwayat keluarga, kebiasaan makan, aktivitas, dan kategori obesitas?
3. **P3:** Model apa yang lebih baik berdasarkan **macro F1 cross-validation**: Regresi Logistik atau Random Forest, dibandingkan baseline prediksi kelas terbanyak?
4. **P4:** Apa dampak menghilangkan **Height** dan **Weight** terhadap kinerja klasifikasi?

### 1.3 Metode dan keputusan evaluasi
- **EDA**: statistik deskriptif, visualisasi target, numerik, dan tabulasi proporsi fitur kategorikal.
- **Cleaning**: cek nilai kosong, duplikasi, tipe data, serta nilai antropometri tidak logis; **tidak** menghapus outlier hanya karena ekstrem.
- **Feature selection berbasis tujuan**: membandingkan dua skenario, yaitu **16 fitur lengkap** dan **14 fitur tanpa Height/Weight**. BMI dibuat **hanya untuk EDA**, tidak diberikan ke model.
- **Modelling**: DummyClassifier (baseline), LogisticRegression, RandomForestClassifier, semuanya memakai *stratified hold-out* 80:20 dan validasi silang 5 lipatan pada **data latih saja**.
- **Pemilihan model terbaik** berdasarkan rerata **macro F1 validasi silang**; data uji digunakan setelah pilihan ditetapkan, termasuk untuk perbandingan deskriptif model pembanding tanpa mengubah keputusan.
- **Evaluasi akhir**: accuracy, macro F1 (utama), weighted F1, balanced accuracy, recall per kelas, confusion matrix, dan permutation importance untuk penjelasan (bukan memilih model).

**Alasan pemilihan macro F1:** tugas memiliki tujuh kelas; rata-rata F1 antar kelas memberi bobot yang sama pada setiap kategori, sehingga lebih informatif dibanding accuracy saja saat ada perbedaan frekuensi antarkelas.

### 1.4 Hipotesis eksploratif
Model dengan Height dan Weight mungkin lebih akurat karena label status berat badan sangat dekat dengan ukuran antropometri. Akan tetapi, model tanpa keduanya lebih bermakna untuk mengeksplorasi pola dari variabel perilaku yang tidak langsung mengodekan kategori berat badan. Hal ini **bukan** bukti hubungan sebab-akibat.

**Prasyarat administratif:** pastikan model Regresi Logistik dan Random Forest termasuk materi praktikum yang diizinkan. Jika tidak, ganti model dengan metode yang sudah diajarkan dan jelaskan alasannya; notebook tidak dapat memverifikasi silabus kelas.


---
# 2. Data Gathering

### 2.1 Asal dan relevansi dataset
Dataset resmi berjudul **Estimation of Obesity Levels Based on Eating Habits and Physical Condition**, berisi **2.111 rekaman, 16 fitur, dan 1 target**, yang dikumpulkan untuk individu dari **Meksiko, Peru, dan Kolombia**. Hal yang perlu dicatat: **sekitar 77% observasi dalam sumber aslinya adalah data sintetis** (WEKA/SMOTE) dan 23% berasal dari pengisian platform web. Karakteristik ini dapat membuat pengujian dalam dataset lebih optimistis dibanding pemakaian pada populasi nyata.

- **Sumber resmi:** https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition
- **DOI dataset:** https://doi.org/10.24432/C5H31Z
- **Artikel pengenalan dataset:** Palechor & de la Hoz Manotas (2019), https://doi.org/10.1016/j.dib.2019.104344
- **Salinan proyek:** https://github.com/puni7676/Klasifikasi-Risiko-Obesitas/blob/main/Obesity_prediction.csv

**Mengapa dipilih:** cocok untuk pembelajaran pengolahan data campuran (numerik dan kategorikal), analisis tujuh kelas, membandingkan model, sekaligus membahas masalah pemilihan fitur dan kemungkinan *shortcut learning*.

### 2.2 Kamus data (nama kolom dalam repository)

| Kolom | Makna | Bentuk / satuan |
|---|---|---|
| `Gender` | Jenis kelamin yang dicatat dalam dataset | Kategorikal |
| `Age` | Umur | Tahun (numerik) |
| `Height` | Tinggi badan | Meter |
| `Weight` | Berat badan | Kilogram |
| `family_history` | Riwayat keluarga dengan kelebihan berat badan | yes/no |
| `FAVC` | Sering mengonsumsi makanan tinggi kalori | yes/no |
| `FCVC` | Frekuensi konsumsi sayuran | Skala perilaku numerik (umumnya 1–3) |
| `NCP` | Jumlah makan utama per hari | Skala/jumlah numerik |
| `CAEC` | Kebiasaan makan di antara jam makan | Kategorikal (no/Sometimes/Frequently/Always) |
| `SMOKE` | Kebiasaan merokok | yes/no |
| `CH2O` | Konsumsi air harian | Skala numerik |
| `SCC` | Memantau konsumsi kalori | yes/no |
| `FAF` | Frekuensi aktivitas fisik | Skala numerik (umumnya 0–3) |
| `TUE` | Waktu menggunakan perangkat teknologi | Skala numerik (umumnya 0–2) |
| `CALC` | Frekuensi konsumsi alkohol | Kategorikal |
| `MTRANS` | Moda transportasi utama | Kategorikal |
| `Obesity` | Kategori obesitas/berat badan (**target**) | 7 kelas |

**Catatan interpretasi:** beberapa nilai skala perilaku dapat berbentuk desimal karena proses data sintetis, sehingga bukan berarti responden melaporkan secara presisi desimal. Nama sumber asli untuk `family_history` dan target dapat berbeda (mis. `family_history_with_overweight`, `NObeyesdad`).

**Lisensi:** CC BY 4.0 (menurut metadata UCI). **Status kesesuaian panduan:** tidak memenuhi persyaratan larangan data sintetis pada Panduan Proyek Akhir 2026/2027, halaman 3, poin 3(f), kecuali ada pengecualian dari pengampu. Tidak bisa mengidentifikasi 23% catatan asli satu per satu dari CSV ini hanya berdasarkan informasi metadata; **jangan menganggap atau merekayasa pemisahan data asli/sintetis**.


In [ ]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display, Markdown

RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')
print('Python:', sys.version.split()[0])
print('Pandas:', pd.__version__, '| NumPy:', np.__version__, '| Scikit-learn:', sklearn.__version__)

# Hanya dataset asli milik proyek; jangan gunakan CSV simulasi untuk menghasilkan hasil tugas.
nama_file = 'Obesity_prediction.csv'
calon_path = [Path.cwd() / nama_file, Path.cwd() / 'data' / nama_file,
              Path.cwd().parent / nama_file, Path.cwd().parent / 'data' / nama_file]
path_csv = next((p for p in calon_path if p.is_file()), None)
sumber_data = str(path_csv) if path_csv is not None else (
    'https://raw.githubusercontent.com/puni7676/Klasifikasi-Risiko-Obesitas/main/Obesity_prediction.csv'
)
try:
    df_raw = pd.read_csv(sumber_data)
except Exception as exc:
    raise RuntimeError(
        'CSV asli tidak terbaca. Simpan Obesity_prediction.csv dari repository di root proyek '
        'atau data/. Jangan mengganti dengan dataset simulasi. Detail: ' + str(exc)
    ) from exc

# Sesuai nama kolom pada repo; terima juga variasi nama header dari sumber UCI.
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={
    'NObeyesdad': 'Obesity',
    'family_history_with_overweight': 'family_history'
})
if len(df_raw) != 2111 or df_raw.shape[1] != 17:
    raise ValueError(
        f'Berkas yang dimuat berukuran {df_raw.shape}, sedangkan CSV sumber proyek '
        '(2.111 baris, 17 kolom). Periksa sumber: jangan gunakan data uji, simulasi, '
        'subset, atau file lain untuk output tugas.'
    )
print('Sumber:', sumber_data)
if path_csv is not None:
    print('SHA-256 CSV lokal:', hashlib.sha256(path_csv.read_bytes()).hexdigest())
else:
    print('CSV dimuat dari GitHub; unduh salinan lokal untuk merekam hash SHA-256 berkas.')
print('Ukuran dataset sebelum cleaning:', df_raw.shape)
display(df_raw.head())

In [ ]:
target = 'Obesity'
kolom_wajib = ['Gender', 'Age', 'Height', 'Weight', 'family_history', 'FAVC', 'FCVC',
               'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', target]
kolom_hilang = sorted(set(kolom_wajib) - set(df_raw.columns))
if kolom_hilang:
    raise ValueError(f'CSV tidak sesuai skema; kolom hilang: {kolom_hilang}')
if df_raw.columns.duplicated().any():
    raise ValueError('Ada nama kolom duplikat; periksa CSV sumber.')

numeric_cols = ['Age','Height','Weight','FCVC','NCP','CH2O','FAF','TUE']
categorical_cols = ['Gender','family_history','FAVC','CAEC','SMOKE','SCC','CALC','MTRANS']
label_order = ['Insufficient_Weight','Normal_Weight','Overweight_Level_I',
               'Overweight_Level_II','Obesity_Type_I','Obesity_Type_II','Obesity_Type_III']

print('Jumlah kolom numerik (prediktor):', len(numeric_cols))
print('Jumlah kolom kategorikal (prediktor):', len(categorical_cols))
print('Jumlah label target:', df_raw[target].nunique(dropna=True))
label_tidak_dikenal = set(df_raw[target].dropna().unique()) - set(label_order)
if label_tidak_dikenal:
    raise ValueError(f'Label target tidak dikenal: {label_tidak_dikenal}')
if df_raw[target].nunique(dropna=True) != 7:
    raise ValueError('Dataset yang dipilih harus memuat ketujuh kategori target.')
print('Duplikasi baris persis:', df_raw.duplicated().sum())
print('Nilai kosong bawaan:', int(df_raw.isna().sum().sum()))
display(df_raw.dtypes.to_frame('Tipe asli'))
display(df_raw[numeric_cols].describe().T.round(2))
print('Frekuensi target:')
display(df_raw[target].value_counts(dropna=False).to_frame('Jumlah'))

**Penjelasan Data Gathering:** tabel awal memperlihatkan jumlah baris/kolom, contoh isi, tipe data, duplikasi, nilai kosong, dan rentang nilai. Angka-angka ini merupakan kondisi **sebelum** cleaning, sehingga menjadi pembanding pada tahap berikutnya. Semua langkah dijalankan terhadap salinan CSV yang benar-benar terbaca, bukan angka hasil rekaan.

---
# 3. Exploratory Data Analysis (EDA)

EDA dilakukan **sebelum** menentukan pembuangan fitur. Visualisasi berikut mencari gambaran distribusi dan asosiasi, bukan membuktikan hubungan kausal. Pemeriksaan awal memakai data mentah untuk mendiagnosis masalah, sedangkan grafik memakai salinan yang dikonversi numeriknya untuk menghindari kesalahan plot.

In [ ]:
df_eda = df_raw.copy()
for col in numeric_cols:
    df_eda[col] = pd.to_numeric(df_eda[col], errors='coerce')

counts = df_eda[target].value_counts().reindex(label_order, fill_value=0)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(counts.index, counts.values)
ax.set(title='Distribusi 7 kategori target', xlabel='Kategori', ylabel='Jumlah responden')
ax.tick_params(axis='x', rotation=40)
for i, v in enumerate(counts):
    ax.text(i, v + counts.max() * 0.01, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()
display(pd.DataFrame({'Jumlah': counts, 'Persentase (%)': (counts / counts.sum() * 100).round(2)}))
if counts.min() > 0:
    display(Markdown(f'**Temuan:** kelas terbanyak adalah **{counts.idxmax().replace("_", " ")}** '
                     f'({counts.max()} orang; {counts.max()/counts.sum():.1%}), sedangkan kelas paling sedikit '
                     f'**{counts.idxmin().replace("_", " ")}** ({counts.min()} orang). '
                     f'Rasio kelas terbanyak:tersedikit = **{counts.max()/counts.min():.2f}:1**. '
                     'Karena itu pemilihan macro F1 dan stratifikasi masuk akal.'))

**Makna grafik distribusi:** distribusi kelas dipakai untuk menilai apakah accuracy saja mencukupi. Pada tugas multikelas, performa per kelas tetap harus dilaporkan, meskipun terlihat hampir seimbang.

In [ ]:
display(df_eda[numeric_cols].describe(percentiles=[.25, .5, .75]).T.round(2))
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(df_eda['Age'].dropna(), bins=25)
ax.set(title='Distribusi usia', xlabel='Usia (tahun)', ylabel='Jumlah observasi')
plt.tight_layout(); plt.show()

usia = df_eda['Age'].dropna()
display(Markdown(
    f'**Temuan grafik usia:** median **{usia.median():.1f} tahun**, '
    f'kuartil bawah (Q1) **{usia.quantile(.25):.1f} tahun**, '
    f'kuartil atas (Q3) **{usia.quantile(.75):.1f} tahun**; '
    f'50% bagian tengah observasi berada antara Q1 dan Q3. Rentang lengkap '
    f'**{usia.min():.1f}–{usia.max():.1f} tahun**. Data usia merupakan deskripsi sampel, '
    'bukan bukti bahwa usia menyebabkan perubahan kategori obesitas.'
))

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
sns.boxplot(data=df_eda, x=target, y='Weight', order=label_order, ax=ax)
ax.set(title='Sebaran berat badan per kategori obesitas', xlabel='Kategori', ylabel='Berat (kg)')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()

# BMI hanya dipakai untuk penjelasan EDA; bukan fitur prediktor.
df_eda['BMI_EDA'] = df_eda['Weight'] / df_eda['Height'].where(df_eda['Height'] > 0).pow(2)
median_bmi = df_eda.groupby(target)['BMI_EDA'].median().reindex(label_order)
median_berat = df_eda.groupby(target)['Weight'].median().reindex(label_order)
display(pd.DataFrame({'Median berat (kg)': median_berat.round(2),
                      'Median BMI (EDA)': median_bmi.round(2)}))

berat_min, berat_max = median_berat.idxmin(), median_berat.idxmax()
display(Markdown(
    f'**Temuan grafik berat:** median berat paling rendah teramati pada '
    f'**{berat_min.replace("_", " ")}** ({median_berat.min():.1f} kg) dan paling tinggi '
    f'pada **{berat_max.replace("_", " ")}** ({median_berat.max():.1f} kg). '
    f'Median BMI antar-kelas berkisar **{median_bmi.min():.1f}–{median_bmi.max():.1f}**. '
    '**Height** dan **Weight** terkait erat dengan penentuan status berat badan; '
    'karena itu, kinerja model lengkap dapat lebih tinggi tanpa membuktikan bahwa '
    'perilaku individu telah berhasil mengidentifikasi risiko secara mandiri. '
    '**BMI_EDA tidak dimasukkan dalam model.**'
))

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
sns.boxplot(data=df_eda, x=target, y='FAF', order=label_order, ax=ax)
ax.set(title='Aktivitas fisik (FAF) per kategori', xlabel='Kategori', ylabel='Skala FAF')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()

med_faf = df_eda.groupby(target)['FAF'].median().reindex(label_order)
med_fcvc = df_eda.groupby(target)['FCVC'].median().reindex(label_order)
display(pd.DataFrame({'Median FAF': med_faf, 'Median FCVC': med_fcvc}).round(2))
display(Markdown(
    f'**Temuan grafik FAF:** median FAF tertinggi **{med_faf.max():.2f}** '
    f'(contoh kelas **{med_faf.idxmax().replace("_", " ")}**) dan terendah '
    f'**{med_faf.min():.2f}** (contoh kelas **{med_faf.idxmin().replace("_", " ")}**). '
    f'Pada variabel konsumsi sayur (FCVC), rentang median antar-kelas adalah '
    f'**{med_fcvc.min():.2f}–{med_fcvc.max():.2f}**. Jika ada kelas dengan median '
    'sama, contoh yang disebut bukan satu-satunya kelas. Selisih ini deskriptif, '
    'bukan bukti sebab-akibat. Nilai pecahan bisa mencerminkan proses sintesis data.'
))

In [ ]:
for col, judul in [
    ('family_history', 'Riwayat keluarga overweight menurut kategori'),
    ('FAVC', 'Konsumsi makanan tinggi kalori menurut kategori'),
    ('CAEC', 'Frekuensi makan di antara jam makan menurut kategori')
]:
    # Normalisasi per baris = membandingkan proporsi DI DALAM masing-masing kelas.
    tab = (pd.crosstab(df_eda[target], df_eda[col], normalize='index')
           .reindex(label_order).fillna(0) * 100)
    ax = tab.plot(kind='bar', stacked=True, figsize=(11,5), width=0.82)
    ax.set(title=judul, xlabel='Kategori', ylabel='Persentase dalam kategori (%)', ylim=(0,100))
    ax.tick_params(axis='x', rotation=40)
    ax.legend(title=col, bbox_to_anchor=(1.01,1), loc='upper left')
    plt.tight_layout(); plt.show()
    display(tab.round(1))
    # Interpretasi dihasilkan dari kolom yang BENAR-BENAR muncul, bukan kalimat generik.
    jawaban = 'yes' if col in ['family_history','FAVC'] and 'yes' in tab.columns else (
        df_eda[col].dropna().value_counts().idxmax()
    )
    proporsi = tab[jawaban]
    display(Markdown(
        f'**Temuan grafik {col}:** untuk respons **{jawaban}**, proporsi tertinggi '
        f'**{proporsi.max():.1f}%** pada kelas **{proporsi.idxmax().replace("_", " ")}** '
        f'dan terendah **{proporsi.min():.1f}%** pada kelas '
        f'**{proporsi.idxmin().replace("_", " ")}** '
        f'(selisih **{proporsi.max()-proporsi.min():.1f} poin persentase**). '
        'Semua persentase dihitung terhadap jumlah observasi dalam kelasnya; '
        'perbedaan tidak menunjukkan hubungan sebab-akibat.'
    ))

In [ ]:
# Korelasi hanya di antara fitur numerik; kategori target TIDAK diubah secara arbitrer menjadi angka.
fig, ax = plt.subplots(figsize=(9,6))
corr = df_eda[numeric_cols].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
ax.set(title='Korelasi Pearson antar prediktor numerik')
plt.tight_layout(); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
         .stack().sort_values(key=lambda s:s.abs(), ascending=False))
if len(pairs):
    (f1, f2), val = pairs.index[0], pairs.iloc[0]
    display(Markdown(f'**Temuan:** korelasi absolut tertinggi teramati antara **{f1}** dan '
                     f'**{f2}** (r = **{val:.2f}**). Korelasi mengukur hubungan linier, '
                     'tidak berarti satu variabel menyebabkan perubahan variabel lain.'))

### 3.1 Rangkuman hasil EDA sebelum preprocessing

Keputusan yang dibawa ke tahap berikutnya: (1) proporsi kategori dan metrik per kelas perlu dipantau; (2) semua fitur numerik diproses sebagai angka, sedangkan fitur kategori dienkode; (3) nilai kosong/duplikat diperiksa terlebih dahulu, bukan menganggap pasti bermasalah; (4) outlier antropometri tidak otomatis dibuang karena variasi berat badan adalah bagian dari masalah; (5) dibutuhkan perbandingan **dengan** dan **tanpa** Height/Weight untuk menguji dampak fitur yang dekat dengan definisi target.

---
# 4. Cleaning, Preprocessing, dan Feature Selection

### 4.1 Pemeriksaan dan cleaning berbasis kondisi data

**Aturan yang diputuskan sebelum pemodelan:** (1) rapikan spasi; (2) samakan penanda kosong seperti `?` dan `NA`; (3) konversi numerik; (4) ubah usia/tinggi/berat yang `<= 0` menjadi missing karena tak masuk akal; (5) buang baris tanpa label, lalu duplikasi baris yang persis sama. Pencilan yang ekstrem **tidak dihapus otomatis** karena harus ada alasan domain, bukan sekadar meningkatkan skor. Missing pada *fitur* ditangani dengan imputasi **dalam pipeline training** agar statistik data uji tidak bocor.

Laporan mencantumkan berapa baris/sel yang berubah **secara nyata**, bukan menciptakan missing value atau duplikasi agar seolah-olah ada cleaning.

In [ ]:
df_clean = df_raw.copy()
sebelum = {
    'Baris': len(df_clean),
    'Kolom': len(df_clean.columns),
    'Duplikat persis': int(df_clean.duplicated().sum()),
    'Missing total (NaN bawaan)': int(df_clean.isna().sum().sum())
}

# Menangani kualitas data asli, tanpa menambah data bermasalah secara buatan.
for col in df_clean.select_dtypes(include=['object', 'string']).columns:
    df_clean[col] = df_clean[col].map(lambda v: v.strip() if isinstance(v, str) else v)
df_clean = df_clean.replace({'': np.nan, '?': np.nan, 'NA': np.nan, 'N/A': np.nan})
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
jumlah_nilai_tidak_logis = int(sum((df_clean[col].dropna() <= 0).sum()
                                for col in ['Age', 'Height', 'Weight']))
for col in ['Age', 'Height', 'Weight']:
    df_clean.loc[df_clean[col] <= 0, col] = np.nan

n_target_missing = int(df_clean[target].isna().sum())
df_clean = df_clean.dropna(subset=[target])
n_duplikat_dibuang = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

sesudah = {
    'Baris': len(df_clean),
    'Kolom': len(df_clean.columns),
    'Duplikat persis': int(df_clean.duplicated().sum()),
    'Missing total (NaN bawaan)': int(df_clean.isna().sum().sum())
}
display(pd.DataFrame({'Sebelum': sebelum, 'Sesudah': sesudah}))
display(pd.DataFrame({
    'Missing pada CSV awal': df_raw.isna().sum(),
    'Missing sesudah cleaning': df_clean.isna().sum()
}))
print('Jumlah nilai Age/Height/Weight tidak logis yang ditandai missing:', jumlah_nilai_tidak_logis)
print('Baris tanpa label dibuang:', n_target_missing)
print('Duplikat persis dibuang (sesudah membuang label kosong):', n_duplikat_dibuang)
print('Total baris berkurang:', len(df_raw) - len(df_clean))
print('Distribusi kelas sesudah cleaning:')
display(df_clean[target].value_counts().reindex(label_order, fill_value=0).to_frame('Jumlah'))

if (df_clean[target].value_counts().reindex(label_order, fill_value=0) < 7).any():
    raise ValueError('Kelas terlalu kecil untuk hold-out stratifikasi dan CV 5-fold; '
                     'periksa hasil cleaning dan sesuaikan metode sebelum melanjutkan.')

display(Markdown(
    f'**Interpretasi cleaning:** jumlah data berubah dari **{len(df_raw)}** menjadi '
    f'**{len(df_clean)}** baris. Sebanyak **{n_target_missing}** baris tanpa label '
    f'dan **{n_duplikat_dibuang}** duplikat persis dibuang; '
    f'**{jumlah_nilai_tidak_logis}** nilai antropometri/usia tak logis ditandai '
    'missing. Nilai missing prediktor yang tersisa diimputasi setelah split, '
    'di setiap fold training. **Jika hasil pemeriksaan menunjukkan nol, tidak '
    'ada perubahan yang diklaim pada bagian tersebut.**'
))

### 4.2 Pemilihan dan rekayasa fitur

| Kelompok | Fitur | Keputusan dan alasan |
|---|---|---|
| Demografi | `Gender`, `Age` | Dipakai dalam kedua skenario sebagai karakteristik individu, tidak sebagai penjelasan kausal. |
| Kebiasaan makan | `FAVC`, `FCVC`, `NCP`, `CAEC`, `CH2O`, `SCC`, `CALC` | Dipakai dalam kedua skenario karena relevan untuk pertanyaan pola kebiasaan. |
| Perilaku, latar keluarga, mobilitas | `family_history`, `SMOKE`, `FAF`, `TUE`, `MTRANS` | Dipakai dalam kedua skenario untuk mempertahankan cakupan variabel dan menguji daya prediksi bersama. |
| Antropometri | `Height`, `Weight` | Hanya pada skenario **lengkap**, dikeluarkan pada **tanpa antropometri** karena dekat dengan konstruksi label. |
| Rekayasa BMI | `BMI_EDA = Weight / Height²` | **Dibuat hanya untuk EDA**; tidak dimasukkan ke fitur model karena mengodekan kategori berat badan secara langsung dan hanya mengulang informasi antropometri. |
| Label | `Obesity` | **Target**; selalu dikeluarkan dari fitur prediktor untuk menghindari kebocoran target yang jelas. |

**Mengapa tidak membuang fitur lain berdasarkan korelasi seluruh data?** Korelasi dengan target atau pentingnya fitur dihitung dari seluruh data berpotensi bocor ke data uji. Pada proyek ini, seleksi fitur sengaja memakai dua konfigurasi **berdasarkan konsep sebelum evaluasi**, sehingga dapat dibandingkan adil. Seluruh transformasi yang *mempelajari parameter* (imputasi, standardisasi, encoding) dilakukan dalam pipeline yang dilatih hanya di *training fold*.

**Perhatian metodologis:** perbandingan pengaruh `Height`/`Weight` dilakukan pada **algoritma yang sama**, sehingga perbedaan skor tidak tercampur oleh perbedaan algoritma. Perbandingan antar-algoritma dilaporkan terpisah.


In [ ]:
# Semua 16 prediktor dibandingkan dengan skenario perilaku + demografi (14 prediktor).
fitur_lengkap = numeric_cols + categorical_cols
fitur_tanpa_antropometri = [c for c in fitur_lengkap if c not in ['Height','Weight']]
feature_sets = {
    'Lengkap (16 fitur)': fitur_lengkap,
    'Tanpa antropometri (14 fitur)': fitur_tanpa_antropometri,
}
assert target not in fitur_lengkap
assert 'BMI_EDA' not in fitur_lengkap
assert len(fitur_lengkap) == 16 and len(fitur_tanpa_antropometri) == 14

display(pd.DataFrame([
    {'Skenario': nama, 'Jumlah fitur': len(cols), 'Fitur': ', '.join(cols)}
    for nama, cols in feature_sets.items()
]))

### 4.3 Split data uji sebelum pembelajaran parameter
Satu pembagian 80% train dan 20% test dengan `stratify=y` digunakan **sama** bagi semua percobaan. Data uji **tidak** digunakan untuk membandingkan atau memilih kandidat model. Proporsi tiap kelas dicek setelah split.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean[fitur_lengkap]
y = df_clean[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Training:', X_train.shape, '| Testing:', X_test.shape)
props_train = y_train.value_counts(normalize=True).reindex(label_order, fill_value=0)
props_test = y_test.value_counts(normalize=True).reindex(label_order, fill_value=0)
display(pd.DataFrame({'Proporsi train (%)': props_train*100,
                      'Proporsi test (%)': props_test*100}).round(2))

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

def buat_preprocessor(cols):
    num = [col for col in cols if col in numeric_cols]
    cat = [col for col in cols if col in categorical_cols]
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num), ('cat', cat_pipe, cat)
    ])

print('Pipeline: numeric → imputasi median → standardisasi;'
      ' kategorikal → modus → one-hot encoding.')
print('Semua parameter dipelajari ulang di setiap training fold CV.')

In [ ]:
# Memperlihatkan contoh fitur SEBELUM dan SESUDAH transformasi.
# Demonstrasi ini hanya FIT pada X_train (tanpa memakai X_test).
print('SEBELUM: contoh 5 baris, fitur belum di-encode')
display(X_train[fitur_lengkap].head())

preprocessor_demo = buat_preprocessor(fitur_lengkap)
X_demo = preprocessor_demo.fit_transform(X_train[fitur_lengkap])
if hasattr(X_demo, 'toarray'):
    preview_data = X_demo[:5].toarray()
else:
    preview_data = X_demo[:5]

all_transformed_columns = preprocessor_demo.get_feature_names_out()
print(f'SESUDAH: {X_train.shape[0]} baris × {X_demo.shape[1]} kolom hasil encoding.')
print('Menampilkan 14 kolom hasil transformasi pertama untuk kemudahan membaca:')
display(pd.DataFrame(preview_data[:, :14],
                     columns=all_transformed_columns[:14]).round(3))
print('Jumlah nilai missing setelah transformasi (seluruh training set):',
      int(np.isnan(X_demo.data).sum() if hasattr(X_demo,'data') and not isinstance(X_demo,np.ndarray)
          else np.isnan(X_demo).sum()))
display(Markdown('**Penjelasan sebelum–sesudah preprocessing:** fitur kategorikal berubah '
                 'menjadi beberapa kolom one-hot; fitur numerik distandardisasi; missing '
                 'ditangani imputer. Demo ini **tidak digunakan untuk melatih model**, '
                 'sehingga setiap fold CV tetap memiliki proses fit preprocessor sendiri.'))

**Alasan transformasi:** Regresi Logistik peka terhadap skala numerik sehingga digunakan `StandardScaler`; Random Forest pada dasarnya tidak memerlukannya, tetapi pipeline yang seragam memudahkan perbandingan yang konsisten. One-hot encoding menghindari pemberian urutan angka palsu pada kategori seperti moda transportasi. Imputasi dipasang di pipeline agar fitting dilakukan hanya pada sampel pelatihan fold terkait.

---
# 5. Modelling

### 5.1 Model dan alasan

- **DummyClassifier (kelas mayoritas)**: baseline; selalu memilih kelas yang paling sering pada data latih. Mewakili model yang tidak memanfaatkan pola prediktor.
- **Regresi Logistik multikelas**: model relatif sederhana untuk membandingkan hubungan antara fitur hasil encoding dan peluang kelas. Standardisasi membantu pelatihannya.
- **Random Forest**: menangkap pola nonlinier dan interaksi tanpa mengasumsikan hubungan linear. Digunakan sebagai model pembanding dengan konfigurasi yang dinyatakan eksplisit, **tanpa menyesuaikan hiperparameter berdasarkan data uji**.

`class_weight` digunakan sebagai pilihan antisipasi perbedaan frekuensi kelas, bukan bukti bahwa dataset pasti timpang. Setiap konfigurasi dijalankan menggunakan pipeline imputasi/encoding dan **StratifiedKFold(5)** pada data latih saja. Pengaturan acak disamakan agar perbandingan dapat direplikasi.

**Kesesuaian silabus:** penggunaan model harus divalidasi terhadap materi praktikum (panduan halaman 3, bagian Pemilihan Model). Jika ada metode di luar materi, metode tersebut harus diganti sebelum penyerahan.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def buat_model(nama):
    if nama == 'Dummy (kelas mayoritas)':
        return DummyClassifier(strategy='most_frequent')
    if nama == 'Regresi Logistik':
        return LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    if nama == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=150, min_samples_leaf=2,
            class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
        )
    raise ValueError('Model tidak dikenali: ' + nama)

rows = []
# Dummy hanya perlu satu skenario karena tidak memanfaatkan fitur.
percobaan = [('Lengkap (16 fitur)', 'Dummy (kelas mayoritas)')]
percobaan += [(skenario, model) for skenario in feature_sets
              for model in ['Regresi Logistik', 'Random Forest']]

for skenario, nama_model in percobaan:
    cols = feature_sets[skenario]
    estimator = Pipeline([
        ('preprocessor', buat_preprocessor(cols)),
        ('classifier', buat_model(nama_model))
    ])
    skor = cross_validate(
        estimator, X_train[cols], y_train, cv=cv,
        scoring={'f1_macro': 'f1_macro',
                 'balanced_accuracy': 'balanced_accuracy',
                 'accuracy': 'accuracy'},
        n_jobs=1, error_score='raise'
    )
    rows.append({
        'Skenario': skenario, 'Model': nama_model,
        'CV F1 macro mean': np.mean(skor['test_f1_macro']),
        'CV F1 macro std': np.std(skor['test_f1_macro']),
        'CV balanced acc': np.mean(skor['test_balanced_accuracy']),
        'CV accuracy': np.mean(skor['test_accuracy'])
    })

cv_results = pd.DataFrame(rows).sort_values('CV F1 macro mean', ascending=False).reset_index(drop=True)
display(cv_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
labels_cv = cv_results['Skenario'] + ' — ' + cv_results['Model']
ax.barh(labels_cv.iloc[::-1], cv_results['CV F1 macro mean'].iloc[::-1],
        xerr=cv_results['CV F1 macro std'].iloc[::-1], capsize=3)
ax.set(title='Perbandingan macro F1 validasi silang (training only)',
       xlabel='Rerata macro F1 ± simpangan baku fold', xlim=(0,1.05))
plt.tight_layout(); plt.show()

non_dummy = cv_results[cv_results['Model'] != 'Dummy (kelas mayoritas)']
terbaik = non_dummy.sort_values(['CV F1 macro mean','CV balanced acc'], ascending=False).iloc[0]
selected_scenario = terbaik['Skenario']
selected_model_name = terbaik['Model']
selected_features = feature_sets[selected_scenario]

baseline_cv = cv_results.loc[cv_results['Model']=='Dummy (kelas mayoritas)', 'CV F1 macro mean'].iloc[0]
display(Markdown(f'**Keputusan dari data latih:** dipilih **{selected_model_name}** pada skenario '
                 f'**{selected_scenario}**, karena CV macro F1-nya paling tinggi '
                 f'(**{terbaik["CV F1 macro mean"]:.3f} ± {terbaik["CV F1 macro std"]:.3f}**). '
                 f'Baseline Dummy hanya **{baseline_cv:.3f}**. '
                 'Nilai dari **test set belum dilihat** saat pemilihan ini.'))

# Perbandingan terbaik per skenario disajikan untuk transparansi.
best_per_scenario = (non_dummy.sort_values('CV F1 macro mean', ascending=False)
                     .groupby('Skenario', sort=False).head(1))
display(best_per_scenario[['Skenario','Model','CV F1 macro mean','CV F1 macro std']].round(4))

# Ablation P4: bandingkan Height/Weight menggunakan algoritma yang SAMA.
baris_ablation = non_dummy[non_dummy['Model'] == selected_model_name].set_index('Skenario')
f1_lengkap = baris_ablation.loc['Lengkap (16 fitur)', 'CV F1 macro mean']
f1_tanpa = baris_ablation.loc['Tanpa antropometri (14 fitur)', 'CV F1 macro mean']
selisih_ablation = f1_lengkap - f1_tanpa
display(Markdown(
    f'**Perbandingan fitur terkontrol:** untuk algoritma **{selected_model_name}** yang sama, '
    f'CV macro F1 dengan 16 fitur = **{f1_lengkap:.3f}**, tanpa Height/Weight = '
    f'**{f1_tanpa:.3f}**. Selisih (lengkap - tanpa) = **{selisih_ablation:+.3f}** poin. '
    'Analisis ini memisahkan perubahan set fitur dari perubahan algoritma.'
))

**Penjelasan pemilihan model:** error bar pada grafik adalah simpangan baku skor dari lima lipatan CV (bukan interval kepercayaan populasi). Model dipilih menggunakan **rerata macro F1 CV**, sebelum mengakses hold-out test. Dua skenario disertakan karena tinggi dan berat sangat berkaitan dengan kategori target. Kinerja yang lebih tinggi dengan kedua fitur tersebut dapat mencerminkan *shortcut learning*, bukan kemampuan membuktikan pengaruh pola hidup. Untuk menjawab efek dua fitur, notebook membandingkan **model algoritma yang sama** di kedua skenario.

---
# 6. Evaluasi Model

### 6.1 Evaluasi akhir pada hold-out

**Keputusan model dikunci terlebih dahulu dari CV data latih.** Model pilihan dilatih ulang pada seluruh data latih dan dievaluasi pada hold-out 20%. Untuk memenuhi syarat membandingkan model pembanding, metrik hold-out untuk semua kandidat juga ditampilkan **hanya sebagai laporan tambahan**. Metrik hold-out tersebut **tidak** digunakan untuk mengganti pemenang.

**Interpretasi metrik:** accuracy adalah persentase prediksi benar; macro F1 merata-ratakan F1 tujuh kelas sehingga kelas kecil tetap mendapat bobot yang sama; weighted F1 memperhitungkan jumlah contoh tiap kelas; balanced accuracy merata-ratakan recall tiap kelas. Skor hanya merefleksikan kemampuan klasifikasi label **dataset ini**, bukan alat diagnosis.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

def hitung_metrik(y_true, y_hat):
    return {
        'Accuracy': accuracy_score(y_true, y_hat),
        'F1 macro': f1_score(y_true, y_hat, labels=label_order,
                             average='macro', zero_division=0),
        'F1 weighted': f1_score(y_true, y_hat, labels=label_order,
                                average='weighted', zero_division=0),
        'Balanced accuracy': balanced_accuracy_score(y_true, y_hat)
    }

# Pemenang tidak diubah setelah melihat test set.
final_model = Pipeline([
    ('preprocessor', buat_preprocessor(selected_features)),
    ('classifier', buat_model(selected_model_name))
])
final_model.fit(X_train[selected_features], y_train)
y_pred = final_model.predict(X_test[selected_features])

baseline_model = DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train, y_train)
y_base = baseline_model.predict(X_test)

# Perbandingan kandidat di test set hanya DESKRIPTIF setelah pilihan CV terkunci.
rows_test = []
for _, kandidat in cv_results.iterrows():
    skenario, nama_model = kandidat['Skenario'], kandidat['Model']
    cols = feature_sets[skenario]
    if nama_model == 'Dummy (kelas mayoritas)':
        pred = y_base
    elif nama_model == selected_model_name and skenario == selected_scenario:
        pred = y_pred
    else:
        model_banding = Pipeline([
            ('preprocessor', buat_preprocessor(cols)),
            ('classifier', buat_model(nama_model))
        ])
        model_banding.fit(X_train[cols], y_train)
        pred = model_banding.predict(X_test[cols])
    rows_test.append({
        'Skenario': skenario,
        'Model': nama_model,
        'Dipilih dari CV': ('Ya' if nama_model == selected_model_name
                          and skenario == selected_scenario else 'Tidak'),
        'CV F1 macro': kandidat['CV F1 macro mean'],
        **hitung_metrik(y_test, pred)
    })

hasil_semua_test = pd.DataFrame(rows_test)
display(hasil_semua_test.round(4))
hasil_test = pd.DataFrame({
    'Model terpilih': hitung_metrik(y_test, y_pred),
    'Dummy baseline': hitung_metrik(y_test, y_base)
}).T

print('Classification report — MODEL TERPILIH (test set):')
print(classification_report(y_test, y_pred, labels=label_order, zero_division=0))
test_acc = hasil_test.loc['Model terpilih', 'Accuracy']
test_f1 = hasil_test.loc['Model terpilih', 'F1 macro']
jumlah_benar = int((y_pred == y_test).sum())
jumlah_test = len(y_test)
display(Markdown(
    f'**Makna angka:** dari **{jumlah_test}** observasi test, '
    f'**{jumlah_benar}** diklasifikasi sesuai label dan '
    f'**{jumlah_test-jumlah_benar}** keliru; accuracy = **{test_acc:.1%}**. '
    f'Macro F1 = **{test_f1:.3f}**, yakni rerata F1 tujuh kelas, **bukan** '
    'persentase penderita yang berhasil didiagnosis. Tabel di atas menunjukkan '
    'kinerja semua model pembanding, tetapi keputusan tetap berdasarkan CV data latih.'
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=label_order)
cm_norm = confusion_matrix(y_test, y_pred, labels=label_order, normalize='true')

fig, ax = plt.subplots(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_order, yticklabels=label_order, ax=ax)
ax.set(title='Confusion matrix (jumlah observasi, test set)',
       xlabel='Prediksi', ylabel='Label asli')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10,7))
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues',
            xticklabels=label_order, yticklabels=label_order, ax=ax,
            vmin=0, vmax=1)
ax.set(title='Confusion matrix per kelas asli (recall per kelas)',
       xlabel='Prediksi', ylabel='Label asli')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()

recall_kelas = pd.Series(np.diag(cm_norm), index=label_order, name='Recall')
display(recall_kelas.to_frame().round(3))
terlemah = recall_kelas.idxmin()
kesalahan = cm.copy()
np.fill_diagonal(kesalahan, 0)
if kesalahan.max() > 0:
    asal_idx, pred_idx = np.unravel_index(kesalahan.argmax(), kesalahan.shape)
    kalimat_kesalahan = (
        f'Kesalahan antar-kelas yang paling sering muncul: '
        f'**{label_order[asal_idx].replace("_", " ")}** diprediksi '
        f'sebagai **{label_order[pred_idx].replace("_", " ")}** '
        f'sebanyak **{kesalahan[asal_idx, pred_idx]}** kali.'
    )
else:
    kalimat_kesalahan = 'Tidak terdapat prediksi lintas kelas yang keliru pada hold-out ini.'
display(Markdown(
    f'**Temuan confusion matrix:** recall terendah kelas '
    f'**{terlemah.replace("_", " ")}** = **{recall_kelas.min():.1%}**; '
    'angka tersebut menunjukkan bagian dari seluruh contoh kelas itu yang berhasil '
    f'dikenali. {kalimat_kesalahan} Perlu memperhatikan jumlah contoh tiap kelas.'
))

In [ ]:
# Permutation importance dihitung sebagai penurunan macro F1 saat kolom diacak.
# Digunakan untuk interpretasi, BUKAN untuk memilih fitur/model setelah test.
from sklearn.inspection import permutation_importance

pi = permutation_importance(
    final_model, X_test[selected_features], y_test,
    scoring='f1_macro', n_repeats=5, random_state=RANDOM_STATE, n_jobs=1
)
importances = pd.DataFrame({
    'Fitur': selected_features,
    'Penurunan F1 macro': pi.importances_mean,
    'Std': pi.importances_std
}).sort_values('Penurunan F1 macro', ascending=False)
display(importances.round(4))

fig, ax = plt.subplots(figsize=(9,6))
plot_pi = importances.sort_values('Penurunan F1 macro')
ax.barh(plot_pi['Fitur'], plot_pi['Penurunan F1 macro'], xerr=plot_pi['Std'], capsize=2)
ax.axvline(0, linestyle='--', linewidth=1)
ax.set(title='Permutation importance pada test set (interpretasi)',
       xlabel='Perubahan macro F1 jika sebuah fitur diacak')
plt.tight_layout(); plt.show()

most_important = importances.iloc[0]
display(Markdown(f'**Temuan:** pengacakan fitur **{most_important["Fitur"]}** '
                 f'menurunkan macro F1 rata-rata sekitar **{most_important["Penurunan F1 macro"]:.3f}** '
                 'pada pengulangan eksperimen ini. Ini menggambarkan ketergantungan '
                 'prediksi pada data, **bukan ukuran pengaruh kausal**. Fitur berkorelasi '
                 'dapat berbagi informasi sehingga importance masing-masing bisa terlihat kecil.'))

### 6.2 Batas interpretasi evaluasi

**Seluruh model pembanding dievaluasi pada data uji hanya setelah kandidat utama dipilih berdasarkan CV**, sehingga tabel hold-out bukan dasar *post-hoc* mengganti pemenang. Perhatikan kesenjangan CV–test dan confusion matrix, bukan mengandalkan accuracy saja. Permutation importance diukur pada hold-out **untuk interpretasi**, bukan memilih fitur lagi. Bila fitur berkorelasi, importance dapat terbagi. Karena 77% observasi sumber sintetis, skor dalam sumber yang sama belum membuktikan kemampuan generalisasi pada pasien/populasi nyata; label data juga berkaitan langsung dengan antropometri.

---
# 7. Konklusi

Kode berikut menyusun jawaban **P1–P4** dari angka **dataset setelah cleaning** serta skor CV/test yang benar-benar dihasilkan. Kesimpulan menampilkan arti angka, pemilihan model, kelemahan per kelas, dan batasan dataset. Output harus **diperiksa dan disimpan** sesudah Run All pada CSV asli. Tidak ada skor/interpretasi numerik yang dibuat-buat sebelum data dieksekusi.

In [ ]:
# Konsisten: seluruh statistik konklusi deskriptif berasal dari df_clean.
n = len(df_clean)
counts_final = df_clean[target].value_counts().reindex(label_order, fill_value=0)
pct_top = counts_final.max() / counts_final.sum() * 100

fh_tab = (pd.crosstab(df_clean[target], df_clean['family_history'], normalize='index')
          .reindex(label_order))
fh_yes = fh_tab['yes'] if 'yes' in fh_tab.columns else None
if fh_yes is not None:
    fh_low = fh_yes.idxmin().replace('_', ' ')
    fh_high = fh_yes.idxmax().replace('_', ' ')
    family_span = (f'dari **{fh_yes.min()*100:.1f}%** pada kelas **{fh_low}** '
                   f'hingga **{fh_yes.max()*100:.1f}%** pada kelas **{fh_high}**')
else:
    family_span = 'tidak dapat dihitung karena kategori `yes` tidak ditemukan'

med_faf_clean = df_clean.groupby(target)['FAF'].median().reindex(label_order)
med_fcvc_clean = df_clean.groupby(target)['FCVC'].median().reindex(label_order)
usia_clean = df_clean['Age'].dropna()

gap = f1_lengkap - f1_tanpa
arah = 'lebih tinggi' if gap > 0 else ('lebih rendah' if gap < 0 else 'sama')

conclusion = f"""## Jawaban pertanyaan penelitian dan keputusan

**P1 — Distribusi dan karakteristik.** Sesudah cleaning tersedia **{n:,}** observasi
({len(df_raw)-n:,} baris lebih sedikit daripada awal) dan **{len(label_order)}** kategori.
Kelas terbanyak adalah **{counts_final.idxmax().replace('_', ' ')}** dengan
**{counts_final.max()}** observasi (**{pct_top:.1f}%**), sedangkan kelas paling sedikit
**{counts_final.idxmin().replace('_', ' ')}** dengan **{counts_final.min()}** observasi.
Median usia adalah **{usia_clean.median():.1f} tahun** (rentang
**{usia_clean.min():.1f}–{usia_clean.max():.1f} tahun**).

**P2 — Pola perilaku dan latar keluarga.** Proporsi jawaban `yes` untuk `family_history`
berbeda antar kelas, {family_span}. Median FAF per kelas berada antara
**{med_faf_clean.min():.2f}–{med_faf_clean.max():.2f}**, sedangkan FCVC antara
**{med_fcvc_clean.min():.2f}–{med_fcvc_clean.max():.2f}**. Pola ini adalah
**asosiasi deskriptif pada dataset**, bukan bukti kausal atau hubungan klinis.

**P3 — Pemilihan model.** Berdasarkan lima lipatan CV pada data latih, pemenang ialah
**{selected_model_name}** dengan **{selected_scenario}**;
macro F1 CV **{terbaik['CV F1 macro mean']:.3f} ± {terbaik['CV F1 macro std']:.3f}**,
dibandingkan dengan Dummy **{baseline_cv:.3f}**. Pada hold-out independen
berukuran **{jumlah_test}**, accuracy **{test_acc:.1%}** berarti
**{jumlah_benar} dari {jumlah_test}** label diprediksi benar;
macro F1 **{test_f1:.3f}** ialah rerata F1 antarkelas, **bukan** tingkat akurasi
diagnosis medis. Recall terendah ialah kelas **{terlemah.replace('_', ' ')}**
(**{recall_kelas.min():.1%}**) sehingga kelas tersebut memerlukan perhatian.
Metrik kandidat lain telah disajikan dalam tabel evaluasi, namun tidak digunakan
untuk mengganti model pemenang CV.

**P4 — Dampak Height/Weight.** Dengan **algoritma {selected_model_name} yang sama**,
macro F1 CV pada 16 fitur adalah **{f1_lengkap:.3f}** dan pada 14 fitur adalah
**{f1_tanpa:.3f}**. Model lengkap **{arah}** sebesar **{abs(gap):.3f} poin F1**
dibandingkan model tanpa antropometri. Perbedaan tersebut tidak otomatis berarti
fitur perilaku mampu memperkirakan risiko klinis; status berat badan berkaitan
erat dengan informasi Height/Weight.

**Keputusan analitis:** pilih pemenang CV sebagai model untuk menjelaskan
klasifikasi **label dataset**; pertahankan analisis ablation tanpa antropometri
sebagai pembanding. **Keputusan administratif belum selesai:** sumber UCI mengandung
~77% observasi sintetis dan bertentangan dengan larangan dataset sintetis dalam
panduan praktikum, sehingga **perlu persetujuan pengampu** untuk digunakan sebagai
proyek akhir.

**Keterbatasan:** (1) mayoritas data sintetis dari SMOTE sehingga data uji dapat
lebih menyerupai data latih daripada kondisi riil; (2) asal data Meksiko, Peru,
dan Kolombia tidak menjamin keterwakilan Indonesia; (3) label sangat terkait
dengan ukuran badan; (4) hanya satu hold-out, tanpa validasi eksternal;
(5) tidak ada analisis sebab-akibat; (6) model dan metrik bukan validasi klinis;
(7) ketersediaan model dalam materi praktikum perlu dikonfirmasi.

**Tindak lanjut:** pastikan dataset dan metode disetujui pengampu;
apabila diperbolehkan, lakukan replikasi dengan data asli yang lebih representatif
dan pelaporan proses pelabelan yang transparan. Jangan mengaku sudah memenuhi
aturan dataset selama masalah data sintetis belum diselesaikan.
"""
display(Markdown(conclusion))

---
# 8. Referensi, reproduksibilitas, dan verifikasi akhir

1. UCI Machine Learning Repository (2019). *Estimation of Obesity Levels Based on Eating Habits and Physical Condition*. https://doi.org/10.24432/C5H31Z. Dataset 2.111 baris, 16 fitur; **77%** observasi dihasilkan sintetis. Lisensi **CC BY 4.0**.
2. Palechor, F. M., & de la Hoz Manotas, A. (2019). *Dataset for estimation of obesity levels based on eating habits and physical condition in individuals from Colombia, Peru and Mexico*. *Data in Brief*. https://doi.org/10.1016/j.dib.2019.104344.
3. Repository kelompok: https://github.com/puni7676/Klasifikasi-Risiko-Obesitas.
4. Dokumentasi Scikit-learn (*cross-validation*): https://scikit-learn.org/stable/modules/cross_validation.html.
5. **Panduan Proyek Akhir Praktikum Data Science 2026/2027**, UPN “Veteran” Yogyakarta, halaman 3–5 (khususnya syarat dataset dan notebook).

**Reproduksibilitas:** `RANDOM_STATE = 42`, *stratified split* 80:20, CV terstratifikasi 5-fold pada train; imputasi, standardisasi, dan encoding berada di dalam pipeline. Ukuran data dan SHA-256 CSV lokal dicetak pada tahap Data Gathering untuk menghindari salah berkas.

### Checklist sebelum dikumpulkan
- [ ] Dataset memenuhi aturan **tidak sintetis** atau sudah ada keputusan resmi pengampu (saat ini **belum terbukti**).
- [ ] Model yang dipakai termasuk materi praktikum.
- [ ] CSV yang dibaca persis salinan repository proyek (2.111 baris sebelum cleaning).
- [ ] **Run All** tidak error; semua grafik, tabel, interpretasi, dan konklusi terlihat **dan disimpan** dalam `.ipynb`.
- [ ] README berisi anggota dan **kontribusi nyata** setiap orang; notebook + CSV juga ada dalam repository.
- [ ] Repo GitHub dikumpulkan mengikuti ketentuan tenggat; commit setelah tenggat tidak dapat dianggap diterima.

**Catatan integritas:** tidak boleh mengarang hasil, memalsukan pembersihan data, atau mengganti CSV asli dengan dataset uji. Notebook bisa dijalankan ulang, tetapi **output numerik final belum dinyatakan selesai sebelum Run All pada CSV asli**.